# Compression-06b — MDL et Kolmogorov en Lean 4 : le cœur combinatoire

Carnet compagnon du lake [`Compression_lean`](lakefile.lean) : il exécute le
module [`Kolmogorov.lean`](Kolmogorov.lean), la formalisation **combinatoire** de
la complexité de Kolmogorov et du principe MDL — **sans** la couche de
calculabilité (pas de machine de Turing, pas de décidabilité). On y montre que
l'inégalité de Kraft (déjà prouvée dans `PrefixCodes.kraft_le_one`) force une
**borne de comptage**, qui à son tour force l'**incompressibilité**, et que la
facture MDL est, à modèle fixé, dominée par les données.

**Lancez les cellules dans l'ordre.** Le kernel `lean4-wsl` charge le lake ;
chaque `#eval` s'exécute sur les définitions du module.

## 0. Charger le lake

In [1]:
import PrefixCodes
import Kolmogorov
open CompressionLean

import PrefixCodes
import Kolmogorov
open CompressionLean
--% env 0

Raw input:
{"cmd": "import PrefixCodes\nimport Kolmogorov\nopen CompressionLean"}
Raw output:
{"env": 0}

## Les résultats, en contrat

Quatre énoncés, du plus général au plus concret — chacun est **prouvé** dans
le module (zéro `sorry`, aucun axiome ad hoc : seulement la fondation
classique standard de Mathlib, cf. annexe), et chaque `#check` / `#eval` de
ce carnet les réexécute :

| Énoncé | Contenu |
|---|---|
| `kraft_short_count` | dans un code préfixe, au plus `2 ^ k` mots de longueur ≤ `k` |
| `atMostK` | au plus `2 ^ k` **données** nommables en ≤ `k` bits |
| `incompressible` | sur `2 ^ n` données, au moins `2 ^ (n-1)` exigent ≥ `n` bits |
| `mdlModelShareTendsToZero` | à modèle fixé, la part du modèle dans la facture MDL → `0` |

## 1. Kraft → borne de comptage

Le théorème central : l'inégalité de Kraft, lue comme un **comptage**. Chaque
mot court (longueur ≤ `k`) consomme au moins `2 ^ (- k)` du budget unitaire de
Kraft, donc il y en a au plus `2 ^ k`.

In [2]:
-- Les longueurs des feuilles du code jouet `sf4` (1, 2, 3, 3).
#eval (leafWords sf4).map List.length

-- Le nombre de mots de longueur ≤ k, pour k = 1, 2, 3.
#eval (shortWords sf4 1).card
#eval (shortWords sf4 2).card
#eval (shortWords sf4 3).card

-- Les longueurs des feuilles du code jouet `sf4` (1, 2, 3, 3).
#eval (leafWords sf4).map List.length
─────▶  [1, 2, 3, 3]

-- Le nombre de mots de longueur ≤ k, pour k = 1, 2, 3.
#eval (shortWords sf4 1).card
─────▶  1
#eval (shortWords sf4 2).card
─────▶  2
#eval (shortWords sf4 3).card
─────▶  4
--% env 1

Raw input:
{"cmd": "-- Les longueurs des feuilles du code jouet `sf4` (1, 2, 3, 3).\n#eval (leafWords sf4).map List.length\n\n-- Le nombre de mots de longueur \u2264 k, pour k = 1, 2, 3.\n#eval (shortWords sf4 1).card\n#eval (shortWords sf4 2).card\n#eval (shortWords sf4 3).card", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "[1, 2, 3, 3]"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "2"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "4"}],
 "env": 1}

In [3]:
-- La borne de comptage, appliquée au code `sf4` (preuves réexécutées ici).
example : (shortWords sf4 1).card ≤ 2 ^ 1 := kraft_short_count sf4 1
example : (shortWords sf4 3).card ≤ 2 ^ 3 := kraft_short_count sf4 3

-- L'arbre complet `full 3` sature exactement la borne : 8 mots, tous de longueur 3.
#eval (shortWords (full 3) 3).card
#eval (shortWords (full 3) 2).card   -- 0 : tous les mots de `full 3` ont longueur 3

-- La borne de comptage, appliquée au code `sf4` (preuves réexécutées ici).
example : (shortWords sf4 1).card ≤ 2 ^ 1 := kraft_short_count sf4 1
example : (shortWords sf4 3).card ≤ 2 ^ 3 := kraft_short_count sf4 3

-- L'arbre complet `full 3` sature exactement la borne : 8 mots, tous de longueur 3.
#eval (shortWords (full 3) 3).card
─────▶  8
#eval (shortWords (full 3) 2).card   -- 0 : tous les mots de `full 3` ont longueur 3
─────▶  0
--% env 2

Raw input:
{"cmd": "-- La borne de comptage, appliqu\u00e9e au code `sf4` (preuves r\u00e9ex\u00e9cut\u00e9es ici).\nexample : (shortWords sf4 1).card \u2264 2 ^ 1 := kraft_short_count sf4 1\nexample : (shortWords sf4 3).card \u2264 2 ^ 3 := kraft_short_count sf4 3\n\n-- L'arbre complet `full 3` sature exactement la borne : 8 mots, tous de longueur 3.\n#eval (shortWords (full 3) 3).card\n#eval (shortWords (full 3) 2).card   -- 0 : tous les mots de `full 3` ont longueur 3", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "8"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "0"}],
 "env": 2}

### Lecture du résultat

La borne est **serrée** sur le code à longueur fixe : `full 3` a exactement
`2 ^ 3 = 8` mots, tous de longueur 3, donc `(shortWords (full 3) 3).card = 8 =
2 ^ 3`. C'est la même saturation de Kraft que le carnet 01b montre pour
`kraftSum` — mais vue du **nombre de mots** plutôt que de la **somme des
poids**. Un code préfixe ne peut pas contenir plus de `2 ^ k` mots courts :
c'est la forme combinatoire du budget unitaire.

## 2. Nommage → borne de comptage des données

On « nomme » une donnée par un mot du code : la machine de nommage
`NamingMachine` est un arbre de code muni d'un décodage `mot → donnée`. Comme
chaque mot code au plus une donnée, les données nommables en ≤ `k` bits ne
dépassent pas les mots courts.

In [4]:
-- Une machine de nommage jouet : le code `sf4` associe à chaque mot la longueur
-- de ce mot (suffisant pour illustrer le comptage des données).
def mNaming : NamingMachine ℕ := { tree := sf4, decode := fun w => w.length }

-- Les données nommables en ≤ k bits, pour k = 1, 2, 3.
#eval ((shortWords mNaming.tree 1).image mNaming.decode).card
#eval ((shortWords mNaming.tree 2).image mNaming.decode).card
#eval ((shortWords mNaming.tree 3).image mNaming.decode).card

-- La borne `atMostK`, appliquée à cette machine.
example : ((shortWords mNaming.tree 2).image mNaming.decode).card ≤ 2 ^ 2 := atMostK mNaming 2

-- Une machine de nommage jouet : le code `sf4` associe à chaque mot la longueur
-- de ce mot (suffisant pour illustrer le comptage des données).
def mNaming : NamingMachine ℕ := { tree := sf4, decode := fun w => w.length }

-- Les données nommables en ≤ k bits, pour k = 1, 2, 3.
#eval ((shortWords mNaming.tree 1).image mNaming.decode).card
─────▶  1
#eval ((shortWords mNaming.tree 2).image mNaming.decode).card
─────▶  2
#eval ((shortWords mNaming.tree 3).image mNaming.decode).card
─────▶  3

-- La borne `atMostK`, appliquée à cette machine.
example : ((shortWords mNaming.tree 2).image mNaming.decode).card ≤ 2 ^ 2 := atMostK mNaming 2
--% env 3

Raw input:
{"cmd": "-- Une machine de nommage jouet : le code `sf4` associe \u00e0 chaque mot la longueur\n-- de ce mot (suffisant pour illustrer le comptage des donn\u00e9es).\ndef mNaming : NamingMachine \u2115 := { tree := sf4, decode := fun w => w.length }\n\n-- Les donn\u00e9es nommables en \u2264 k bits, pour k = 1, 2, 3.\n#eval ((shortWords mNaming.tree 1).image mNaming.decode).card\n#eval ((shortWords mNaming.tree 2).image mNaming.decode).card\n#eval ((shortWords mNaming.tree 3).image mNaming.decode).card\n\n-- La borne `atMostK`, appliqu\u00e9e \u00e0 cette machine.\nexample : ((shortWords mNaming.tree 2).image mNaming.decode).card \u2264 2 ^ 2 := atMostK mNaming 2", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "2"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "3"}],
 "env": 3}

### Lecture du résultat

Sur `sf4`, les données nommables en ≤ 1 bit sont au plus `2 ^ 1 = 2`, en ≤ 2
bits au plus `4`, en ≤ 3 bits au plus `8`. Le décodage `decode` peut
**fusionner** plusieurs mots courts vers une même donnée (l'image ne dépasse
jamais le nombre de mots), donc la borne des **mots** (`kraft_short_count`)
borne les **données** (`atMostK`) — c'est exactement le
`Finset.card_image_le` que la preuve invoque.

In [5]:
-- Exercice 1.
-- TODO etudiant : notez vos prédictions AVANT d'exécuter.
-- Combien de données la machine `mNaming` nomme-t-elle en ≤ 2 bits ? En ≤ 3 bits ?
#eval ((shortWords mNaming.tree 2).image mNaming.decode).card
#eval ((shortWords mNaming.tree 3).image mNaming.decode).card

-- Exercice 1.
-- TODO etudiant : notez vos prédictions AVANT d'exécuter.
-- Combien de données la machine `mNaming` nomme-t-elle en ≤ 2 bits ? En ≤ 3 bits ?
#eval ((shortWords mNaming.tree 2).image mNaming.decode).card
─────▶  2
#eval ((shortWords mNaming.tree 3).image mNaming.decode).card
─────▶  3
--% env 4

Raw input:
{"cmd": "-- Exercice 1.\n-- TODO etudiant : notez vos pr\u00e9dictions AVANT d'ex\u00e9cuter.\n-- Combien de donn\u00e9es la machine `mNaming` nomme-t-elle en \u2264 2 bits ? En \u2264 3 bits ?\n#eval ((shortWords mNaming.tree 2).image mNaming.decode).card\n#eval ((shortWords mNaming.tree 3).image mNaming.decode).card", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "2"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "3"}],
 "env": 4}

## 3. Incompressibilité

Le corollaire le plus parlant : sur `2 ^ n` données, on ne peut pas les
compresser **toutes**. Au moins la moitié exigent ≥ `n` bits.

In [6]:
-- L'incompressibilité (énoncé ; la preuve est dans le module).
#check incompressible

-- Concrètement, pour n = 4 : 2 ^ 4 = 16 données, au moins 2 ^ 3 = 8 exigent ≥ 4 bits.
#eval (2 : ℕ) ^ 4
#eval (2 : ℕ) ^ 3

-- L'incompressibilité (énoncé ; la preuve est dans le module).
#check incompressible
──────▶  CompressionLean.incompressible {α : Type} [DecidableEq α] (m : NamingMachine α) (n : ℕ) (hn : 0 < n) (D : Finset α)
  (hD : D.card = 2 ^ n) : {x ∈ D | ¬namedWithin m x (n - 1)}.card ≥ 2 ^ (n - 1)

-- Concrètement, pour n = 4 : 2 ^ 4 = 16 données, au moins 2 ^ 3 = 8 exigent ≥ 4 bits.
#eval (2 : ℕ) ^ 4
─────▶  16
#eval (2 : ℕ) ^ 3
─────▶  8
--% env 5

Raw input:
{"cmd": "-- L'incompressibilit\u00e9 (\u00e9nonc\u00e9 ; la preuve est dans le module).\n#check incompressible\n\n-- Concr\u00e8tement, pour n = 4 : 2 ^ 4 = 16 donn\u00e9es, au moins 2 ^ 3 = 8 exigent \u2265 4 bits.\n#eval (2 : \u2115) ^ 4\n#eval (2 : \u2115) ^ 3", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "CompressionLean.incompressible {α : Type} [DecidableEq α] (m : NamingMachine α) (n : ℕ) (hn : 0 < n) (D : Finset α)\n  (hD : D.card = 2 ^ n) : {x ∈ D | ¬namedWithin m x (n - 1)}.card ≥ 2 ^ (n - 1)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "16"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "8"}],
 "env": 5}

### Lecture du résultat

C'est le **théorème d'incompressibilité** en forme combinatoire : si `2 ^ n`
données pouvaient toutes être nommées en ≤ `n - 1` bits, il y aurait au moins
`2 ^ n` données nommables en ≤ `n - 1` bits — mais `atMostK` borne celles-ci à
`2 ^ (n-1)`, contradiction. Donc au moins `2 ^ n - 2 ^ (n-1) = 2 ^ (n-1)`
données exigent ≥ `n` bits. **On ne peut pas tout compresser** : avec un bit de
moins, la moitié des données ne rentre pas.

In [7]:
-- Exercice 2.
-- TODO etudiant : pour n = 6, combien de données (sur 2 ^ 6) exigent ≥ 6 bits ?
-- (indice : la borne est 2 ^ (n-1))
#eval (2 : ℕ) ^ 6
#eval (2 : ℕ) ^ 5

-- Exercice 2.
-- TODO etudiant : pour n = 6, combien de données (sur 2 ^ 6) exigent ≥ 6 bits ?
-- (indice : la borne est 2 ^ (n-1))
#eval (2 : ℕ) ^ 6
─────▶  64
#eval (2 : ℕ) ^ 5
─────▶  32
--% env 6

Raw input:
{"cmd": "-- Exercice 2.\n-- TODO etudiant : pour n = 6, combien de donn\u00e9es (sur 2 ^ 6) exigent \u2265 6 bits ?\n-- (indice : la borne est 2 ^ (n-1))\n#eval (2 : \u2115) ^ 6\n#eval (2 : \u2115) ^ 5", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "64"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "32"}],
 "env": 6}

## 4. La facture MDL

Le principe MDL minimise `mdlCost = description du modèle + description des
données`. À **modèle fixé**, quand les données grossissent, la part du modèle
dans la facture devient arbitrairement petite.

In [8]:
-- La facture MDL : modèle + données (en bits).
#eval (mdlCost 10 100 : ℕ)
#eval (mdlCost 10 1000 : ℕ)

-- La part du modèle : m / (m + n). Pour m = 10, elle s'efface quand n grossit.
#eval ((10 : ℚ) / (10 + 100))
#eval ((10 : ℚ) / (10 + 1000))

-- Le théorème : pour tout k ≥ 1, au-delà d'un certain N, m / (m + n) ≤ 1 / k.
#check mdlModelShareTendsToZero

-- La facture MDL : modèle + données (en bits).
#eval (mdlCost 10 100 : ℕ)
─────▶  110
#eval (mdlCost 10 1000 : ℕ)
─────▶  1010

-- La part du modèle : m / (m + n). Pour m = 10, elle s'efface quand n grossit.
#eval ((10 : ℚ) / (10 + 100))
─────▶  1 / 11
#eval ((10 : ℚ) / (10 + 1000))
─────▶  1 / 101

-- Le théorème : pour tout k ≥ 1, au-delà d'un certain N, m / (m + n) ≤ 1 / k.
#check mdlModelShareTendsToZero
──────▶  CompressionLean.mdlModelShareTendsToZero (m k : ℕ) : 0 < k → ∃ N, ∀ n ≥ N, ↑m / (↑m + ↑n) ≤ 1 / ↑k
--% env 7

Raw input:
{"cmd": "-- La facture MDL : mod\u00e8le + donn\u00e9es (en bits).\n#eval (mdlCost 10 100 : \u2115)\n#eval (mdlCost 10 1000 : \u2115)\n\n-- La part du mod\u00e8le : m / (m + n). Pour m = 10, elle s'efface quand n grossit.\n#eval ((10 : \u211a) / (10 + 100))\n#eval ((10 : \u211a) / (10 + 1000))\n\n-- Le th\u00e9or\u00e8me : pour tout k \u2265 1, au-del\u00e0 d'un certain N, m / (m + n) \u2264 1 / k.\n#check mdlModelShareTendsToZero", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "110"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "1010"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "1 / 11"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "1 / 101"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "CompressionLean.mdlModelShareTendsToZero (m k : ℕ) : 0 < k → ∃ N, ∀ n ≥ N, ↑m / (↑m + ↑n) ≤ 1 / ↑k"}],
 "env": 7}

### Lecture du résultat

`mdlCost 10 100 = 110` et `mdlCost 10 1000 = 1010` : les `10` bits du modèle
sont **noyés** dans les données. La part du modèle `10 / 110 ≈ 0.091` tombe à
`10 / 1010 ≈ 0.010` quand `n` passe de `100` à `1000`. Le théorème
`mdlModelShareTendsToZero` en fait une limite : la part du modèle tend vers `0`,
donc **à modèle fixé, c'est le terme « données » qui gouverne la facture** — le
coût de décrire le modèle est un terme d'ordre inférieur, amorti par la taille
des données.

In [9]:
-- Exercice 3.
-- TODO etudiant : pour m = 20 bits de modèle, calculez la part du modèle
-- m / (m + n) pour n = 200 puis n = 2000. Observez la décroissance.
#eval ((20 : ℚ) / (20 + 200))
#eval ((20 : ℚ) / (20 + 2000))

-- Exercice 3.
-- TODO etudiant : pour m = 20 bits de modèle, calculez la part du modèle
-- m / (m + n) pour n = 200 puis n = 2000. Observez la décroissance.
#eval ((20 : ℚ) / (20 + 200))
─────▶  1 / 11
#eval ((20 : ℚ) / (20 + 2000))
─────▶  1 / 101
--% env 8

Raw input:
{"cmd": "-- Exercice 3.\n-- TODO etudiant : pour m = 20 bits de mod\u00e8le, calculez la part du mod\u00e8le\n-- m / (m + n) pour n = 200 puis n = 2000. Observez la d\u00e9croissance.\n#eval ((20 : \u211a) / (20 + 200))\n#eval ((20 : \u211a) / (20 + 2000))", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "1 / 11"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "1 / 101"}],
 "env": 8}

## 5. Annexe — quels axiomes ?

Les preuves n'emploient **aucun axiome ad hoc** ni aucun `sorry` :
`#print axioms` liste *exactement* les axiomes atteints par chaque preuve.
Ce que la liste révèle — et c'est le cas de la quasi-totalité des énoncés de
Mathlib — sont les trois axiomes de **fondation classique standard** :
`propext` (extensionnalité des propositions), `Classical.choice` (choix
classique) et `Quot.sound` (cohérence des quotients). Ils proviennent de
l'infrastructure de Mathlib (lemmes sur les `Finset`, décidabilité), pas du
raisonnement combinatoire lui-même.

In [10]:
#print axioms kraft_short_count
#print axioms atMostK
#print axioms incompressible
#print axioms mdlModelShareTendsToZero

#print axioms kraft_short_count
──────▶  'CompressionLean.kraft_short_count' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms atMostK
──────▶  'CompressionLean.atMostK' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms incompressible
──────▶  'CompressionLean.incompressible' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms mdlModelShareTendsToZero
──────▶  'CompressionLean.mdlModelShareTendsToZero' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 9

Raw input:
{"cmd": "#print axioms kraft_short_count\n#print axioms atMostK\n#print axioms incompressible\n#print axioms mdlModelShareTendsToZero", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data":
   "'CompressionLean.kraft_short_count' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'CompressionLean.atMostK' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'CompressionLean.incompressible' depends on axioms: [propext, Classical.choice, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "'CompressionLean.mdlModelShareTendsToZero' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 9}

### Lecture du résultat

Chaque `#print axioms` répond de la même liste :
`[propext, Classical.choice, Quot.sound]`. Ce sont les trois axiomes de
**fondation classique** de Mathlib, présents dans les dépendances de la
quasi-totalité de sa bibliothèque — pas des axiomes introduits par ces
preuves. Le raisonnement en lui-même reste **constructif** (arithmétique sur
`ℕ` et `ℚ`, inégalités de puissances de `2`, comptage de `Finset`) : la
borne de comptage, l'incompressibilité et la limite MDL n'invocent **aucun**
choix non constructif au-delà de la fondation standard.

## À retenir

- **Kraft = comptage** : `kraft_le_one` (carnet 01b) force `kraft_short_count`
  — un code préfixe compte au plus `2 ^ k` mots de longueur ≤ `k`.
- **Comptage = borne des données** : `atMostK` — les données nommables en ≤ `k`
  bits ne dépassent pas les mots courts.
- **Incompressibilité** : `incompressible` — sur `2 ^ n` données, au moins
  `2 ^ (n-1)` exigent ≥ `n` bits. On ne peut pas tout compresser.
- **MDL** : `mdlCost = modèle + données` ; à modèle fixé,
  `mdlModelShareTendsToZero` — la part du modèle tend vers `0`.
- **Fondation** : `#print axioms` ne montre que `[propext, Classical.choice,
  Quot.sound]` sur les quatre énoncés — la fondation classique standard de
  Mathlib ; aucun axiome ad hoc, zéro `sorry`.

Le module [`Kolmogorov.lean`](Kolmogorov.lean) est le **cœur combinatoire** du
jumeau Lean de la série Compression (EPIC #18706) ; il **importe**
`PrefixCodes.kraft_le_one` (maison canonique de l'inégalité de Kraft) sans la
redéfinir.